In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, MultiHeadAttention, Dense, Dropout, Add, LayerNormalization
from tensorflow.keras.layers import Layer, Lambda, Input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.activations import softmax
import gc

# to reduce memory
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Masking for Attention

In [2]:
def create_padding_mask(token_ids):
    """
    Create a padding mask for self-attention in Transformer layers.

    The mask will have shape (batch_size, 1, 1, seq_len) to allow proper broadcasting in MultiHeadAttention.
    Each dimension has a specific purpose:

        batch_size   : keeps a separate mask for each sequence in the batch.
        1 (num_heads): broadcast the same mask across all attention heads.
        1 (query)    : broadcast the same mask to all query positions in the sequence.
        seq_len      : indicates which key positions (tokens) should be masked (padding).

    Masking is applied to the key positions only, making sure that query tokens do not attend to padding tokens,
    while it doesn’t matter if padding tokens themselves attend to other positions.

    Args:
        token_ids (tf.Tensor): Token ID matrix of shape (batch_size, seq_len),

    Returns:
        mask: A 4D float mask of shape (batch_size, 1, 1, seq_len),
            where 1.0 indicates a valid token, 0 represents padding tokens.
    """

    mask = 1 - tf.cast(tf.equal(token_ids, 0), tf.float32)  # (batch_size, seq_len)
    return mask[:, tf.newaxis, tf.newaxis, :]

In [3]:
x = tf.constant(
    [[7., 6., 0., 0., 1.],
    [1., 2., 3., 0., 0.],
    [0., 0., 0., 4., 5.]]
)

padding_mask = create_padding_mask(x)
padding_mask

<tf.Tensor: shape=(3, 1, 1, 5), dtype=float32, numpy=
array([[[[1., 1., 0., 0., 1.]]],


       [[[1., 1., 1., 0., 0.]]],


       [[[0., 0., 0., 1., 1.]]]], dtype=float32)>

In [4]:
def create_causal_mask(target_seq_len):
    """
    Create a causal (look-ahead) mask for decoder self-attention as 0/1 float.
    The mask prevents each query token from attending to future key tokens.

    It has shape (1, 1, target_seq_len, target_seq_len) to allow broadcasting across batch size and attention heads.
    Each dimension has a specific purpose:

        1 (batch_size)        : broadcast the same mask for all sequences in the batch.
        1 (num_heads)         : broadcast the same mask across all attention heads.
        target_seq_len (query): each row represents a query token position.
        target_seq_len (key)  : each column represents a key token position.

    Future positions (j > i) are masked with 0, while positions j <= i are attendable (1).

    Args:
        target_seq_len (int): Length of the target sequence.

    Returns:
        tf.Tensor: A float32 mask tensor of shape (1, 1, seq_len, seq_len)
    """

    # lower triangular matrix of ones
    mask = tf.linalg.band_part(tf.ones((target_seq_len, target_seq_len), dtype=tf.float32), -1, 0)
    return mask[tf.newaxis, tf.newaxis, :, :]

In [5]:
causal_mask = create_causal_mask(5)
causal_mask

<tf.Tensor: shape=(1, 1, 5, 5), dtype=float32, numpy=
array([[[[1., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 0.],
         [1., 1., 1., 1., 1.]]]], dtype=float32)>

In [6]:
def compute_attention_output(query, key, value, mask=None):
    """
    Scaled Dot-Product Attention (Transformer standard)
    This function computes attention outputs and weights according to:
        Attention(Q, K, V) = softmax((Q * K^T) / sqrt(d_qk)) * V

    Args:
        query: Tensor of shape (batch_size, num_heads, seq_len_q, depth_qk)
            - Query vectors for each token.

        key: Tensor of shape (batch_size, num_heads, seq_len_kv, depth_qk)
            - Key vectors to be compared against queries.

        value: Tensor of shape (batch_size, num_heads, seq_len_kv, depth_v)
            - Value vectors containing the information to be aggregated.

        mask: (optional) Tensor of shape (batch_size, num_heads, seq_len_q, seq_len_kv)
              - Used to mask out padding tokens or future tokens (in decoder).
              - Masked positions will be set to -inf before the softmax step,
                causing their attention weights to become zero.

    Returns:
        attention_scores: Tensor of shape (batch_size, num_heads, seq_len_q, depth_v)
            - The output after applying attention weights to the value vectors.

        attention_weights: Tensor of shape (batch_size, num_heads, seq_len_q, seq_len_kv)
            - The normalized attention distributions for each query token.

    Notes:
        - seq_len_q and seq_len_kv may differ in cross-attention between output sequence and input sequence.
        - depth_qk and depth_v can differ, but Q and K must share the same depth for Q @ K^T to be valid.
        - Typically: (NOT ALWAYS THIS!)
              depth = d_model / num_heads
          where d_model is the total embedding dimension.
    """

    qk = tf.matmul(query, key, transpose_b=True)  # (batch_size, num_heads, seq_len_q, seq_len_kv)
    depth_qk = tf.cast(tf.shape(query)[-1], tf.float32)

    attention_logits = qk / tf.math.sqrt(depth_qk)

    if mask is not None:
        # valid_token = 0, padding_token = -1e9
        mask = (1.0 - mask) * (-1e9)
        # logits for valid tokens remain the same, logits for padding tokens will be ~ -1e9, softmax(-1e9) = 0
        attention_logits += mask

    attention_scores = tf.nn.softmax(attention_logits, axis=-1)  # (batch_size, num_heads, seq_len_q, seq_len_kv)

    attention_output = tf.matmul(attention_weights, value)  # (batch_size, num_heads, seq_len_q, depth_v)

    return attention_output, attention_scores

# Encoder

In [7]:
class EncoderLayer(Layer):
    def __init__(self, embedding_dim, n_attn_heads, ffn_hidden_dim, dropout_rate=0.1, layernorm_eps=1e-6, **kwargs):
        super().__init__(**kwargs)

        # Self-Attention
        self.self_attn = MultiHeadAttention(
            num_heads=n_attn_heads,
            key_dim=embedding_dim // n_attn_heads,  # key_dim = query_dim
            value_dim=embedding_dim // n_attn_heads,
            dropout=dropout_rate,  # dropout after compute attention_weights
            name=f"{self.name}_self_attention"
        )

        # Feed Forward
        self.ffn = Sequential([
            Dense(ffn_hidden_dim, activation='relu'),
            Dense(embedding_dim)
        ], name=f'{self.name}_feed_forward_network')

        # Dropout
        self.dropout1 = Dropout(dropout_rate, name=f'{self.name}_dropout_after_attn')
        self.dropout2 = Dropout(dropout_rate, name=f'{self.name}_dropout_after_ffn')

        # Residual + LayerNorm
        self.add1 = Add(name=f'{self.name}_add_after_attn')
        self.add2 = Add(name=f'{self.name}_add_after_ffn')
        self.layernorm1 = LayerNormalization(epsilon=layernorm_eps, name=f'{self.name}_layernorm_after_attn')
        self.layernorm2 = LayerNormalization(epsilon=layernorm_eps, name=f'{self.name}_layernorm_after_ffn')


    def call(self, x, padding_mask=None, training=False):
        """
        Forward pass of a Transformer Encoder Layer.

        Parameters:
            x: tf.Tensor of shape (batch_size, seq_len, embedding_dim)
                Input embedding sequence.

            padding_mask: tf.Tensor, optional, shape (batch_size, 1, 1, seq_len)
                Mask to prevent attention to padding tokens.

            training: bool, default=False
                If True, the layer is in training mode and dropout layers will be applied.
                If False, the layer is in inference mode and dropout will be disabled.

        Returns:
            tf.Tensor of shape (batch_size, seq_len, embedding_dim)
                Output of the Encoder Layer after:
                - Multi-Head Self-Attention with Dropout, Residual connection, and Layer Normalization
                - Feed-Forward Network with Dropout, Residual connection, and Layer Normalization
        """

        # Self-Attention
        attn_output = self.self_attn(query=x, key=x, value=x, attention_mask=padding_mask, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.add1([x, attn_output])
        out1 = self.layernorm1(out1)

        # Feed Forward
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.add2([out1, ffn_output])
        out2 = self.layernorm2(out2)

        return out2


    def compute_output_shape(self, input_shape):
        # Output shape same as input, to show output shape in this layer in model.summary()
        return input_shape

In [8]:
class PositionalEncodingLayer(Layer):
    def __init__(self, seq_length, embedding_dim, **kwargs):
        super().__init__(**kwargs)
        self.pe = tf.constant(self._encode_position(seq_length, embedding_dim), dtype=tf.float32)  # make it constant, not for training

    def _encode_position(self, seq_length, embedding_dim):
        positions = np.arange(seq_length)[:, np.newaxis]  # shape (seq_len, 1)
        dims = np.arange(embedding_dim)[np.newaxis, :]  # shape (1, embedding_dim)
        denominator = np.power(
            10000,
            (2*(dims//2)) / np.float32(embedding_dim)
        )
        pe = positions / denominator  # shape (seq_len, embedding_dim)
        pe[:, 0::2] = np.sin(pe[:, 0::2])  # start from 0, step 2 --> 0,2,4...
        pe[:, 1::2] = np.cos(pe[:, 1::2])  # start from 1, step 2 --> 1,3,5,...
        return pe[np.newaxis, ...]  # (1, seq_len, embedding_dim)

    def call(self, x):
        """
        x: (batch_size, seq_len, embedding_dim)
        """
        batch_size, seq_len = tf.shape(x)[0], tf.shape(x)[1]
        pe = self.pe[:, :seq_len, :]
        pe = tf.tile(pe, [batch_size, 1, 1])
        return tf.cast(pe, x.dtype)

    def compute_output_shape(self, input_shape):
        return input_shape

In [9]:
class EncoderModel(Model):
    def __init__(self, n_layers, seq_length, vocab_size, embedding_dim, n_attn_heads, ffn_hidden_dim, dropout_rate=0.01, layernorm_eps=1e-6, **kwargs):
        super().__init__(**kwargs)

        self.embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, name='embedding_layer')
        self.pe_layer = PositionalEncodingLayer(seq_length, embedding_dim, name='positional_encoding_layer')

        self.scaler = Lambda(lambda x: x * tf.math.sqrt(tf.cast(embedding_dim, x.dtype)), name='scale_after_embedding')

        self.add0 = Add(name='add_before_encoder')

        self.dropout0 = Dropout(dropout_rate, name='dropout_before_encoder')

        self.encoder_layers = [
            EncoderLayer(embedding_dim, n_attn_heads, ffn_hidden_dim, dropout_rate, layernorm_eps, name=f'encoder_layer_{i}')
            for i in range(n_layers)
        ]

    def call(self, x, padding_mask=None, training=False):
        """
        Forward pass of the Transformer Encoder model.

        Parameters:
            x: tf.Tensor of shape (batch_size, seq_len)
                Input token IDs (integer indices from the vocabulary).

            padding_mask: tf.Tensor, optional, shape (batch_size, 1, 1, seq_len)
                Mask to prevent attention to padding tokens.

            training: bool, default=False
                Whether the model is in training mode (dropout active).

        Returns:
            tf.Tensor of shape (batch_size, seq_len, embedding_dim)
                Encoded contextual representations for each token position.
        """

        x_emb = self.embedding_layer(x)
        x_emb = self.scaler(x_emb)

        x = self.add0([x_emb, self.pe_layer(x_emb)])

        x = self.dropout0(x, training=training)

        for layer in self.encoder_layers:
            x = layer(x, padding_mask=padding_mask, training=training)

        return x

# BERT (Encoder-Only Version)

In [10]:
def create_bert(model_size="base", seq_length=512, vocab_size=30522, dropout_rate=0.1):

    if model_size.lower() == "base":
        n_layers = 12
        embedding_dim = 768
        n_attn_heads = 12
        ffn_hidden_dim = 3072
        print("Using BERT-Base configuration")

    elif model_size.lower() == "large":
        n_layers = 24
        embedding_dim = 1024
        n_attn_heads = 16
        ffn_hidden_dim = 4096
        print("Using BERT-Large configuration")

    elif model_size == "distil":
        n_layers = 6
        embedding_dim = 768
        n_attn_heads = 12
        ffn_hidden_dim = 3072
        print("Using DistilBERT configuration")

    else:
        raise ValueError("model_size must be one of ['base', 'large', 'distil']")

    # Initialize the Encoder model
    encoder = EncoderModel(
        n_layers=n_layers,
        seq_length=seq_length,
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        n_attn_heads=n_attn_heads,
        ffn_hidden_dim=ffn_hidden_dim,
        dropout_rate=dropout_rate,
        name=f"Bert_{model_size.capitalize()}"
    )

    return encoder

In [11]:
batch_size = 8
seq_length = 512
vocab_size = 30522

input = tf.random.uniform(
    shape=(batch_size, seq_length),
    minval=0,
    maxval=vocab_size,
    dtype=tf.int32
)

model_sizes = ["base", "large", "distil"]
for size in model_sizes:
    # avoid OOM
    tf.keras.backend.clear_session()
    gc.collect()

    model = create_bert(size)
    output = model(input)
    model.summary()
    print()
    print('~'*100)

    # avoid OOM
    del model
    gc.collect()


Using BERT-Base configuration


Model: "Bert_Base"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (8, 512, 768)          │    23,440,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_layer       │ (8, 512, 768)          │             0 │
│ (PositionalEncodingLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scale_after_embedding (Lambda)  │ (8, 512, 768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_before_encoder (Add)        │ (8, 512, 768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_before_encoder          │ ?                      │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_0 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_1 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_2 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_3 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_4 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_5 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_6 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_7 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_8 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_9 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_10 (EncoderLayer) │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_11 (EncoderLayer) │ (8, 512, 768)          │     7,087,872 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 108,495,360 (413.88 MB)

 Trainable params: 108,495,360 (413.88 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (8, 512, 768)          │    23,440,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_layer       │ (8, 512, 768)          │             0 │
│ (PositionalEncodingLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scale_after_embedding (Lambda)  │ (8, 512, 768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_before_encoder (Add)        │ (8, 512, 768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_before_encoder          │ ?                      │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_0 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_1 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_2 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_3 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_4 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_layer_5 (EncoderLayer)  │ (8, 512, 768)          │     7,087,872 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,968,128 (251.65 MB)

 Trainable params: 65,968,128 (251.65 MB)

 Non-trainable params: 0 (0.00 B)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


# Decoder

In [12]:
class DecoderLayer(Layer):
    def __init__(self, embedding_dim, n_attn_heads, ffn_hidden_dim,
                 dropout_rate=0.1, layernorm_eps=1e-6, use_cross_attn=True, **kwargs):
        super().__init__(**kwargs)

        self.use_cross_attn = use_cross_attn

        # Masked Self-Attention
        self.masked_self_attn = MultiHeadAttention(
            num_heads=n_attn_heads,
            key_dim=embedding_dim // n_attn_heads,
            value_dim=embedding_dim // n_attn_heads,
            dropout=dropout_rate,
            name=f"{self.name}_masked_self_attention"
        )

        # Encoder-Decoder Attention (optional)
        if self.use_cross_attn:
            self.cross_attn = MultiHeadAttention(
                num_heads=n_attn_heads,
                key_dim=embedding_dim // n_attn_heads,
                value_dim=embedding_dim // n_attn_heads,
                dropout=dropout_rate,
                name=f"{self.name}_cross_attention"
            )

        # Feed Forward
        self.ffn = Sequential([
            Dense(ffn_hidden_dim, activation='relu'),
            Dense(embedding_dim)
        ], name=f'{self.name}_feed_forward_network')

        # Dropout
        self.dropout1 = Dropout(dropout_rate, name=f"{self.name}_dropout_after_self_attn")
        self.dropout2 = Dropout(dropout_rate, name=f"{self.name}_dropout_after_cross_attn")
        self.dropout3 = Dropout(dropout_rate, name=f"{self.name}_dropout_after_ffn")

        # Residual + LayerNorm
        self.add1 = Add(name=f"{self.name}_add_after_self_attn")
        self.add2 = Add(name=f"{self.name}_add_after_cross_attn")
        self.add3 = Add(name=f"{self.name}_add_after_ffn")
        self.layernorm1 = LayerNormalization(epsilon=layernorm_eps, name=f"{self.name}_layernorm_after_self_attn")
        self.layernorm2 = LayerNormalization(epsilon=layernorm_eps, name=f"{self.name}_layernorm_after_cross_attn")
        self.layernorm3 = LayerNormalization(epsilon=layernorm_eps, name=f"{self.name}_layernorm_after_ffn")


    def build(self, input_shape):
        if self.use_cross_attn:
            self.cross_attn.build(query_shape=input_shape, key_shape=input_shape, value_shape=input_shape)

        self.masked_self_attn.build(query_shape=input_shape, key_shape=input_shape, value_shape=input_shape)

        self.ffn.build(input_shape)

        self.layernorm1.build(input_shape)
        self.layernorm2.build(input_shape)
        self.layernorm3.build(input_shape)

        self.built = True


    def call(self, x, encoder_output=None, causal_mask=None, decoder_padding_mask=None, encoder_padding_mask=None, training=False):
        """
        Forward pass of a Transformer Decoder Layer.

        Parameters:
            x: tf.Tensor of shape (batch_size, target_seq_len, embedding_dim)
                Input embeddings for the decoder.

            encoder_output: tf.Tensor of shape (batch_size, source_seq_len, embedding_dim)
                Output from the encoder to attend to (optional for decoder-only models).

            causal_mask: tf.Tensor, optional, shape (1, 1, target_seq_len, target_seq_len)
                Mask to prevent attention to future tokens (for autoregressive decoding).

            decoder_padding_mask: tf.Tensor, optional, shape (batch_size, 1, 1, target_seq_len)
                Mask to prevent self-attention on padding tokens in decoder input.

            encoder_padding_mask: tf.Tensor, optional, shape (batch_size, 1, 1, source_seq_len)
                Mask to prevent attention to padding tokens in encoder output (optional for decoder-only models).

            training: bool, default=False
                Whether the model is in training mode (applies dropout).

        Returns:
            tf.Tensor of shape (batch_size, target_seq_len, embedding_dim)
                Decoder output embeddings.
        """

        if decoder_padding_mask is not None:
            final_mask = tf.logical_or(decoder_padding_mask, causal_mask)  # (batch_size, 1, target_seq_len, target_seq_len)
        else:
            final_mask = causal_mask

        # Masked Self-Attention
        attn1 = self.masked_self_attn(query=x, key=x, value=x, attention_mask=final_mask, training=training)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.add1([x, attn1])
        x = self.layernorm1(out1)

        # Encoder-Decoder Attention (Optional)
        if self.use_cross_attn and encoder_output is not None and encoder_padding_mask is not None:
            attn2 = self.cross_attn(query=x, key=encoder_output, value=encoder_output, attention_mask=encoder_padding_mask, training=training)
            attn2 = self.dropout2(attn2, training=training)
            out2 = self.add2([x, attn2])
            x = self.layernorm2(out2)

        # Feed Forward
        ffn_output = self.ffn(x)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.add3([x, ffn_output])
        x = self.layernorm3(out3)

        return x


    def compute_output_shape(self, input_shape):
        return input_shape

In [13]:
class DecoderModel(Model):
    def __init__(self, n_layers, target_seq_length, target_vocab_size, embedding_dim, n_attn_heads, ffn_hidden_dim,
                 dropout_rate=0.01, layernorm_eps=1e-6, use_cross_attn=True, **kwargs):
        super().__init__(**kwargs)

        self.embedding_layer = Embedding(input_dim=target_vocab_size, output_dim=embedding_dim, name='embedding_layer')
        self.pe_layer = PositionalEncodingLayer(target_seq_length, embedding_dim, name='positional_encoding_layer')

        self.scaler = Lambda(lambda x: x * tf.math.sqrt(tf.cast(embedding_dim, x.dtype)), name='scale_after_embedding')

        self.add0 = Add(name='add_before_decoder')

        self.dropout0 = Dropout(dropout_rate, name='dropout_before_decoder')

        self.decoder_layers = [
            DecoderLayer(
                embedding_dim, n_attn_heads, ffn_hidden_dim, dropout_rate, layernorm_eps,
                use_cross_attn=use_cross_attn, name=f'decoder_layer_{i}'
            )
            for i in range(n_layers)
        ]


    def call(self, x, encoder_output=None, causal_mask=None, decoder_padding_mask=None, encoder_padding_mask=None, training=False):
        """
        Forward pass for the Transformer Decoder model.

        Parameters:
            x: tf.Tensor of shape (batch_size, target_seq_len)
                Input token IDs for the decoder.

            encoder_output: tf.Tensor of shape (batch_size, source_seq_len, embedding_dim)
                Output from the encoder to attend to (optional for decoder-only models).

            causal_mask: tf.Tensor, optional, shape (1, 1, target_seq_len, target_seq_len)
                Mask to prevent attention to future tokens (for autoregressive decoding).

            decoder_padding_mask: tf.Tensor, optional, shape (batch_size, 1, 1, target_seq_len)
                Mask to prevent self-attention on padding tokens in decoder input.

            encoder_padding_mask: tf.Tensor, optional, shape (batch_size, 1, 1, source_seq_len)
                Mask to prevent attention to padding tokens in encoder output (optional for decoder-only models).

            training: bool, default=False
                Whether the model is in training mode (applies dropout).

        Returns:
            tf.Tensor of shape (batch_size, target_seq_len, embedding_dim)
                The decoder output embeddings after all decoder layers.
        """

        x_emb = self.embedding_layer(x)
        x_emb = self.scaler(x_emb)

        x = self.add0([x_emb, self.pe_layer(x_emb)])

        x = self.dropout0(x, training=training)

        for layer in self.decoder_layers:
            x = layer(
                x, encoder_output,
                causal_mask=causal_mask,
                decoder_padding_mask=decoder_padding_mask,
                encoder_padding_mask=encoder_padding_mask,
                training=training
            )
        return x

# GPT (Decoder-Only Version)

In [14]:
def create_gpt(model_version="davinci-175b", target_seq_length=2048, vocab_size=50257, dropout_rate=0.1):
    model_version = model_version.lower()

    if model_version == "ada":
        n_layers = 12
        embedding_dim = 768
        n_attn_heads = 12
        ffn_hidden_dim = 3072
        print("Using GPT-3 Ada (125M) configuration")

    elif model_version == "babbage":
        n_layers = 24
        embedding_dim = 1024
        n_attn_heads = 16
        ffn_hidden_dim = 4096
        print("Using GPT-3 Babbage (350M) configuration")

    elif model_version == "curie":
        n_layers = 24
        embedding_dim = 1536
        n_attn_heads = 16
        ffn_hidden_dim = 6144
        print("Using GPT-3 Curie (760M) configuration")

    elif model_version == "davinci-1.3b":
        n_layers = 24
        embedding_dim = 2048
        n_attn_heads = 32
        ffn_hidden_dim = 8192
        print("Using GPT-3 Davinci (1.3B) configuration")

    elif model_version == "davinci-2.7b":
        n_layers = 32
        embedding_dim = 2560
        n_attn_heads = 32
        ffn_hidden_dim = 10240
        print("Using GPT-3 Davinci (2.7B) configuration")

    elif model_version == "davinci-6.7b":
        n_layers = 32
        embedding_dim = 4096
        n_attn_heads = 32
        ffn_hidden_dim = 16384
        print("Using GPT-3 Davinci (6.7B) configuration")

    elif model_version == "davinci-13b":
        n_layers = 40
        embedding_dim = 5120
        n_attn_heads = 40
        ffn_hidden_dim = 20480
        print("Using GPT-3 Davinci (13B) configuration")

    elif model_version == "davinci-175b":
        n_layers = 96
        embedding_dim = 12288
        n_attn_heads = 96
        ffn_hidden_dim = 49152
        print("Using GPT-3 Davinci (175B) configuration")

    else:
        raise ValueError("model_version must be one of GPT-3 versions")

    # Initialize decoder-only model
    decoder = DecoderModel(
        n_layers=n_layers,
        target_seq_length=target_seq_length,
        target_vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        n_attn_heads=n_attn_heads,
        ffn_hidden_dim=ffn_hidden_dim,
        dropout_rate=dropout_rate,
        use_cross_attn=False,
        name=f"{model_version.upper()}"
    )

    return decoder

In [15]:
batch_size = 1
seq_length = 128  # reduce this to reduce OOM
vocab_size = 50257

input = tf.random.uniform(
    shape=(batch_size, seq_length),
    minval=0,
    maxval=vocab_size,
    dtype=tf.int32
)

gpt_version = ["ada", "babbage", "curie", "davinci-1.3b", "davinci-2.7b"]  # that's the limit of GPT T4
for version in gpt_version:
    # avoid OOM
    tf.keras.backend.clear_session()
    gc.collect()

    model = create_gpt(version, target_seq_length=seq_length)
    output = model(input)
    model.summary()
    print()
    print('~'*100)

    # avoid OOM
    del model
    gc.collect()


Using GPT-3 Ada (125M) configuration


Model: "ADA"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (1, 128, 768)          │    38,597,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_layer       │ (1, 128, 768)          │             0 │
│ (PositionalEncodingLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scale_after_embedding (Lambda)  │ (1, 128, 768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_before_decoder (Add)        │ (1, 128, 768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_before_decoder          │ ?                      │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_0 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_1 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_2 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_3 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_4 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_5 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_6 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_7 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_8 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_9 (DecoderLayer)  │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_10 (DecoderLayer) │ (1, 128, 768)          │     7,089,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_11 (DecoderLayer) │ (1, 128, 768)          │     7,089,408 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 123,670,272 (471.76 MB)

 Trainable params: 123,670,272 (471.76 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ (1, 128, 2560)         │   128,657,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_layer       │ (1, 128, 2560)         │             0 │
│ (PositionalEncodingLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scale_after_embedding (Lambda)  │ (1, 128, 2560)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_before_decoder (Add)        │ (1, 128, 2560)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_before_decoder          │ ?                      │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_0 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_1 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_2 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_3 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_4 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_5 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_6 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_7 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_8 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_9 (DecoderLayer)  │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_10 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_11 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_12 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_13 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_14 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_15 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_16 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_17 (DecoderLayer) │ (1, 128, 2560)         │    78,681,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_layer_18 (DecoderLayer) │ (1, 128, 2560)         │    78,681,60

 Total params: 2,646,469,120 (9.86 GB)

 Trainable params: 2,646,469,120 (9.86 GB)

 Non-trainable params: 0 (0.00 B)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


# Transformer (Encoder-Decoder Version)

In [16]:
class Transformer(Model):
    def __init__(
        self,
        n_encoder_layers, n_decoder_layers,
        src_seq_length, tgt_seq_length,
        src_vocab_size, tgt_vocab_size,
        embedding_dim,
        n_attn_heads,
        ffn_hidden_dim,
        dropout_rate=0.1,
        layernorm_eps=1e-6,
        **kwargs
    ):
        super().__init__(**kwargs)

        # --- Encoder ---
        self.encoder = EncoderModel(
            n_layers=n_encoder_layers,
            seq_length=src_seq_length,
            vocab_size=src_vocab_size,
            embedding_dim=embedding_dim,
            n_attn_heads=n_attn_heads,
            ffn_hidden_dim=ffn_hidden_dim,
            dropout_rate=dropout_rate,
            layernorm_eps=layernorm_eps,
            name="encoder"
        )

        # --- Decoder ---
        self.decoder = DecoderModel(
            n_layers=n_decoder_layers,
            target_seq_length=tgt_seq_length,
            target_vocab_size=tgt_vocab_size,
            embedding_dim=embedding_dim,
            n_attn_heads=n_attn_heads,
            ffn_hidden_dim=ffn_hidden_dim,
            dropout_rate=dropout_rate,
            layernorm_eps=layernorm_eps,
            use_cross_attn=True,
            name="decoder"
        )

        # --- Final Linear ---
        self.final_linear = Dense(tgt_vocab_size, name="output_projection")


    def call(
        self,
        encoder_input,
        decoder_input,
        encoder_padding_mask=None,
        decoder_padding_mask=None,
        causal_mask=None,
        training=False
    ):
        """
        Forward pass of Transformer (Encoder-Decoder).

        Parameters:
            encoder_input: (batch_size, src_seq_len)
            decoder_input: (batch_size, tgt_seq_len)
            encoder_padding_mask: (batch_size, 1, 1, src_seq_len)
            decoder_padding_mask: (batch_size, 1, 1, tgt_seq_len)
            causal_mask: (1, 1, tgt_seq_len, tgt_seq_len)
            training: bool

        Returns:
            logits: (batch_size, tgt_seq_len, tgt_vocab_size)
        """

        encoder_output = self.encoder(
            encoder_input, padding_mask=encoder_padding_mask, training=training
        )

        decoder_output = self.decoder(
            decoder_input,
            encoder_output=encoder_output,
            causal_mask=causal_mask,
            decoder_padding_mask=decoder_padding_mask,
            encoder_padding_mask=encoder_padding_mask,
            training=training
        )

        logits = self.final_linear(decoder_output)
        return logits

# T5 / BART

In [17]:
def create_t5(
    model_version="t5-base",
    src_seq_length=512,
    tgt_seq_length=512,
    src_vocab_size=32128,
    tgt_vocab_size=32128,
    dropout_rate=0.1
):
    model_version = model_version.lower()

    if model_version == "t5-small":
        n_layers = 6
        embedding_dim = 512
        n_attn_heads = 8
        ffn_hidden_dim = 2048
        print("Using T5-Small (60M) configuration")

    elif model_version == "t5-base":
        n_layers = 12
        embedding_dim = 768
        n_attn_heads = 12
        ffn_hidden_dim = 3072
        print("Using T5-Base (220M) configuration")

    elif model_version == "t5-large":
        n_layers = 24
        embedding_dim = 1024
        n_attn_heads = 16
        ffn_hidden_dim = 4096
        print("Using T5-Large (770M) configuration")

    elif model_version == "t5-3b":
        n_layers = 24
        embedding_dim = 1024
        n_attn_heads = 32
        ffn_hidden_dim = 16384
        print("Using T5-3B configuration")

    elif model_version == "t5-11b":
        n_layers = 24
        embedding_dim = 1024
        n_attn_heads = 128
        ffn_hidden_dim = 65536
        print("Using T5-11B configuration")

    else:
        raise ValueError("Unknown T5 version. Must be one of: t5-small, t5-base, t5-large, t5-3b, t5-11b")

    # Initialize full encoder-decoder Transformer
    model = Transformer(
        n_encoder_layers=n_layers,
        n_decoder_layers=n_layers,
        src_seq_length=src_seq_length,
        tgt_seq_length=tgt_seq_length,
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        embedding_dim=embedding_dim,
        n_attn_heads=n_attn_heads,
        ffn_hidden_dim=ffn_hidden_dim,
        dropout_rate=dropout_rate,
        layernorm_eps=1e-6,
        name=model_version.upper()
    )

    return model

In [18]:
def create_bart(
    model_version="bart-base",
    src_seq_length=1024,
    tgt_seq_length=1024,
    src_vocab_size=50265,
    tgt_vocab_size=50265,
    dropout_rate=0.1
):
    model_version = model_version.lower()

    if model_version == "bart-base":
        n_layers = 6
        embedding_dim = 768
        n_attn_heads = 12
        ffn_hidden_dim = 3072
        print("Using BART-Base (140M) configuration")

    elif model_version == "bart-large":
        n_layers = 12
        embedding_dim = 1024
        n_attn_heads = 16
        ffn_hidden_dim = 4096
        print("Using BART-Large (400M) configuration")

    elif model_version == "mbart-50":
        n_layers = 12
        embedding_dim = 1024
        n_attn_heads = 16
        ffn_hidden_dim = 4096
        src_vocab_size = tgt_vocab_size = 250000  # overwrite
        print("Using mBART-50 (610M) multilingual configuration")

    elif model_version == "distilbart":
        n_layers = 6
        embedding_dim = 768
        n_attn_heads = 12
        ffn_hidden_dim = 3072
        print("Using DistilBART (90M) configuration")

    else:
        raise ValueError("Unknown BART version. Must be one of: bart-base, bart-large, mbart-50, distilbart")

    # Initialize full encoder-decoder Transformer
    model = Transformer(
        n_encoder_layers=n_layers,
        n_decoder_layers=n_layers,
        src_seq_length=src_seq_length,
        tgt_seq_length=tgt_seq_length,
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        embedding_dim=embedding_dim,
        n_attn_heads=n_attn_heads,
        ffn_hidden_dim=ffn_hidden_dim,
        dropout_rate=dropout_rate,
        layernorm_eps=1e-5,
        name=model_version.upper()
    )

    return model